# ECG-FM Stage 2 — Kaggle Version

### Before running this notebook:

**1. Add PTB-XL dataset** (right sidebar → Add Data → search 'ptb-xl-dataset' by khyeh0719)

**2. Upload your model files as a Kaggle dataset:**
- Go to kaggle.com → Datasets → New Dataset
- Name it exactly: `ecgfm-models`
- Upload two files from your local `models/` folder:
  - `mimic_iv_ecg_physionet_pretrained.pt` (from `models/ecgfm/`)
  - `ecgfm_stage1.pt` (from `models/`)
- Then add it to this notebook: Add Data → Your Datasets → ecgfm-models

**3. Enable GPU:** right sidebar → Accelerator → GPU T4 x2

**4. Run all cells in order**

---
Output model `ecgfm_stage2.pt` will appear in the Output tab automatically.

In [ ]:
# Cell 1 — Check GPU and RAM
import torch, psutil

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU — enable it in the right sidebar before continuing')

print('System RAM:', round(psutil.virtual_memory().total / 1e9, 1), 'GB')

In [ ]:
# Cell 2 — Find PTB-XL and model files paths
import os
from pathlib import Path

print('Available datasets:')
for d in sorted(os.listdir('/kaggle/input')):
    print(f'  /kaggle/input/{d}/')

# Auto-detect PTB-XL path
PTBXL_KAGGLE = None
for d in os.listdir('/kaggle/input'):
    candidate = f'/kaggle/input/{d}'
    if os.path.exists(f'{candidate}/ptbxl_database.csv'):
        PTBXL_KAGGLE = candidate
        break

if PTBXL_KAGGLE:
    print(f'\nPTB-XL found at: {PTBXL_KAGGLE}')
    n_dat = len(list(Path(PTBXL_KAGGLE).rglob('*.dat')))
    print(f'Signal files: {n_dat}')
    if n_dat < 20000:
        print('WARNING: fewer than expected .dat files — check the dataset has records500/')
else:
    raise FileNotFoundError('PTB-XL not found. Add the ptb-xl-dataset via Add Data in the right sidebar.')

# Auto-detect model files path
MODEL_KAGGLE = None
for d in os.listdir('/kaggle/input'):
    candidate = f'/kaggle/input/{d}'
    if os.path.exists(f'{candidate}/ecgfm_stage1.pt'):
        MODEL_KAGGLE = candidate
        break

if MODEL_KAGGLE:
    print(f'\nModel files found at: {MODEL_KAGGLE}')
    for f in ['mimic_iv_ecg_physionet_pretrained.pt', 'ecgfm_stage1.pt']:
        size = round(os.path.getsize(f'{MODEL_KAGGLE}/{f}') / 1e6)
        print(f'  {f}: {size} MB')
else:
    raise FileNotFoundError('Model files not found. Upload ecgfm-models dataset and add it via Add Data.')

In [ ]:
# Cell 3 — Clone repo and install dependencies
import os

WORKDIR = '/kaggle/working/EKG'

if not os.path.exists(WORKDIR):
    from getpass import getpass
    token = getpass('GitHub personal access token (repo is private): ')
    !git clone https://{token}@github.com/shlomih/EKG.git {WORKDIR}
else:
    print('Repo already cloned')

%cd {WORKDIR}
!pip install -q wfdb scipy scikit-learn
print('Setup done')

In [ ]:
# Cell 4 — Link PTB-XL and model files into project structure
import os, shutil

# Symlink PTB-XL dataset → path expected by ecgfm_finetune.py
os.makedirs('ekg_datasets', exist_ok=True)
link = 'ekg_datasets/ptbxl'
if os.path.islink(link):
    os.remove(link)
os.symlink(PTBXL_KAGGLE, link)
print(f'Linked: {link} -> {PTBXL_KAGGLE}')

# Copy model checkpoints into project
os.makedirs('models/ecgfm', exist_ok=True)
shutil.copy(f'{MODEL_KAGGLE}/mimic_iv_ecg_physionet_pretrained.pt',
            'models/ecgfm/mimic_iv_ecg_physionet_pretrained.pt')
shutil.copy(f'{MODEL_KAGGLE}/ecgfm_stage1.pt',
            'models/ecgfm_stage1.pt')

print('Model files copied:')
print(' ', round(os.path.getsize('models/ecgfm/mimic_iv_ecg_physionet_pretrained.pt')/1e6), 'MB  pretrained encoder')
print(' ', round(os.path.getsize('models/ecgfm_stage1.pt')/1e6), 'MB  stage 1 checkpoint')
print('\nReady for training.')

In [ ]:
# Cell 5 — Run Stage 2 fine-tuning
#
# Expected: ~20 min/epoch on T4, ~3 hours total (early stopping patience=8)
# Checkpoint saved to models/ecgfm_stage2.pt at every improvement
#
# If CUDA out-of-memory: change --batch_s2 8 to --batch_s2 4
# torch.set_num_threads is already set to 2 in ecgfm_finetune.py

!python -u ecgfm_finetune.py --stage 2 --batch_s2 8

In [ ]:
# Cell 6 — Show results
# The model is saved under /kaggle/working/ and will appear in the Output tab
import os, torch

model_path = 'models/ecgfm_stage2.pt'
if os.path.exists(model_path):
    size_mb = round(os.path.getsize(model_path) / 1e6)
    print(f'ecgfm_stage2.pt saved ({size_mb} MB)')
    print('Download it from the Output tab on the right sidebar.\n')

    ckpt = torch.load(model_path, map_location='cpu', weights_only=False)
    tm   = ckpt.get('test_metrics', {})
    print('Final test results:')
    print(f"  Accuracy : {tm.get('acc', 0):.1%}")
    print(f"  HYP F1   : {tm.get('hyp_f1', 0):.3f}  (baseline 0.375)")
    print(f"  Macro F1 : {tm.get('macro_f1', 0):.3f}  (baseline 0.492)")
else:
    print('WARNING: ecgfm_stage2.pt not found — training may not have completed')

## After training

**Download `ecgfm_stage2.pt`:**
- Right sidebar → Output tab → `EKG/models/ecgfm_stage2.pt` → download
- Save it to your local `models/` folder

### What to expect
| Metric | Stage 1 (frozen) | Target Stage 2 |
|---|---|---|
| HYP F1 | 0.375 | > 0.50 |
| Macro F1 | 0.492 | > 0.70 |

If Stage 2 beats the current multi-label CNN (MacroAUROC=0.972, MacroF1=0.699), it replaces the CNN backbone.
See `DECISIONS.md` D-006 and D-007 for context.